In [0]:
# Gold layer: dim_ip_geo (Kimball IP-geolocation dimension, Type 1
#         current-state) - gold enrichment #2 (IP geo, MUST-DECIDE #5).
# Source: b2b_ecommerce.silver.logs - one row per DISTINCT non-null
#         client_ip (dims-from-silver pattern; fact_web_logs.client_ip
#         is copied verbatim from silver.logs, so KPI views join the
#         fact to this dim on the shared client_ip business key).
#         Country resolves via LEFT JOIN to b2b_ecommerce.gold.ref_ip_geo
#         (21-row standalone reference, first_octet PK -> ISO-2
#         country_code; one-time S3 load, NOT a pipeline dataset - a
#         stable reference read at run time).
# Why Type 1 (KPI-driven, requirements.md "Required KPIs"):
#         KPI 10 (Geo Traffic: traffic by country) needs IP->country
#         IDENTITY only - no KPI uses geo history, so SCD2 would be
#         dead weight here.
# Grain: one row per client_ip (business key + MERGE key + join key
#         from fact_web_logs). NULL client_ip rows are excluded (AUTO
#         CDC keys must be non-null; silver currently has 0 nulls but
#         the guard stays defensive). If a NULL ever appeared, those
#         fact rows would simply not match - same 'unknown' surfacing
#         as below.
# Resolution rule (ref_ip_geo's stated design): an IPv4 whose first
#         octet is NOT in the reference, or any non-IPv4 address,
#         resolves to UNKNOWN - by design, never force-located. This
#         dim stores that as NULL country_code (an ISO-2 column must
#         not hold a fake value); KPI views COALESCE to 'unknown'. The
#         two unknown flavors stay distinguishable via first_octet:
#         NULL = non-IPv4, non-NULL-but-unmapped = IPv4 outside the
#         reference. Pre-flight (verified against ALL distinct IPs):
#           75,809 distinct non-null client_ip
#             70,180 resolved (92.6%)
#             4,274 IPv4-unmapped  -> NULL country
#             1,355 non-IPv4      -> NULL country, NULL first_octet
#         Log-grain baseline for post-build verification (totals to
#         538,152 = all fact rows): AR 264,296 / US 94,791 /
#         BR 35,071 / CL 29,965 / unmapped-IPv4 26,468 / UY 13,280 /
#         CN 12,082 / FR 11,881 / DE 11,823 / IN 11,106 / ES 10,129 /
#         MX 8,927 / non-IPv4 8,333.
# Columns: ip_sk + client_ip + first_octet + country_code - the
#         KPI-trimmed set. city/region/ASN do not exist in the
#         reference (first-octet granularity is all the data
#         supports); log_rows deliberately NOT stored (a property of
#         the fact, changes every load, would break snapshot-diff
#         idempotency).
# Surrogate key: ip_sk = crc32(client_ip) - deterministic, always
#         non-negative and SHORT (unsigned 32-bit; user-directed
#         crc32 convention), stable across refreshes. Pre-flight
#         verified 0 collisions across all 75,809 IPs (checked at
#         this cardinality precisely because a 32-bit hash can
#         collide; views join on client_ip, not ip_sk, so even a
#         future collision would not corrupt a join - re-verify per
#         build).
# MERGE / idempotency: AUTO CDC FROM SNAPSHOT into a Delta-backed
#         streaming table, keyed on client_ip (requirements' MERGE
#         constraint), Type 1 - an identical snapshot diffs to zero
#         events, so retries and no-op runs change nothing. An IP that
#         disappears from silver is deleted here too (snapshot diff
#         semantics); a REF change re-resolves country on the next
#         run (Type 1 update in place).
# Quality: no expectations - silver owns row-level validation, and
#         expectations are not allowed on AUTO CDC FROM SNAPSHOT flows
#         anyway (same decision as the other dims).
# Layout: not partitioned (dims are not date-partitioned),
#         liquid-clustered by ip_sk (dims cluster by their surrogate
#         key).
from pyspark import pipelines as dp
from pyspark.sql import functions as F

SILVER_LOGS = "b2b_ecommerce.silver.logs"
REF_IP_GEO = "b2b_ecommerce.gold.ref_ip_geo"

# Dotted-quad IPv4 shape (all four octets 1-3 digits; an out-of-range
# octet like 999 still extracts an int that simply matches no
# reference row - never force-located).
IPV4_PATTERN = "^[0-9]{1,3}([.][0-9]{1,3}){3}$"


# Silver's complete current-state IP snapshot, first octet extracted,
# then country resolved via LEFT JOIN to the reference (NULL =
# unknown by design). A batch temporary view: the engine reads it at
# each pipeline run (ingestion-time snapshot versioning). No
# driver-side actions anywhere. The select doubles as the column trim.
@dp.temporary_view(name="ip_geo_snapshot")
def ip_geo_snapshot():
    ip = F.col("client_ip")
    ips = (
        spark.read.table(SILVER_LOGS)
        .where(ip.isNotNull())
        .select("client_ip")
        .distinct()
        # non-IPv4 addresses extract no octet -> NULL first_octet
        .withColumn(
            "first_octet",
            # split_part args must be Columns in PySpark - a bare "."
            # or 1 is parsed as a column name (INVALID_ATTRIBUTE_NAME)
            F.when(
                ip.rlike(IPV4_PATTERN),
                F.split_part(ip, F.lit("."), F.lit(1)).cast("int"),
            ),
        )
    )
    ref = spark.read.table(REF_IP_GEO).select(
        F.col("first_octet").alias("ref_octet"), "country_code"
    )
    return (
        ips.join(ref, ips["first_octet"] == ref["ref_octet"], "left")
        .drop("ref_octet")
        .withColumn("ip_sk", F.crc32(ip))
        .select("ip_sk", "client_ip", "first_octet", "country_code")
    )


# Delta-backed streaming table: the AUTO CDC target, clustered on the
# surrogate key (dims cluster by their surrogate key).
dp.create_streaming_table(
    name="dim_ip_geo",
    comment=(
        "Kimball IP-geolocation dimension, SCD Type 1 (current state): one row "
        "per distinct non-null client_ip, MERGE-loaded from the "
        "b2b_ecommerce.silver.logs current-state snapshot via AUTO CDC FROM "
        "SNAPSHOT keyed on client_ip - identical snapshots diff to zero events, "
        "so retries are idempotent; IPs removed from silver are deleted here. "
        "Country resolves via first-octet LEFT JOIN to ref_ip_geo (21-row "
        "reference); an unmapped first octet or a non-IPv4 address stays NULL "
        "country_code - unknown by design, never force-located (KPI views "
        "COALESCE to 'unknown'). Deterministic surrogate key ip_sk = "
        "crc32(client_ip) (unsigned 32-bit, non-negative, short, 0 collisions "
        "across all 75,809 IPs at build time). SCD2 deliberately not applied "
        "(no KPI needs geo history); city/region do not exist at first-octet "
        "reference granularity."
    ),
    cluster_by=["ip_sk"],
)


# The auto-merge load: diff each snapshot against the last committed one
# and apply inserts/updates/deletes in place (Type 1 - no history
# columns).
dp.create_auto_cdc_from_snapshot_flow(
    target="dim_ip_geo",
    source="ip_geo_snapshot",  # ingestion-time snapshot versioning
    keys=["client_ip"],
    stored_as_scd_type=1,
)